#### **FACTIBILIDAD DE LA SOLUCIÓN**

In [343]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from importlib import reload

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")

Guardamos primero los tipos de cajas y productos en listas de python.

In [344]:
caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")

cajas = [
    Caja(
        caja_id = row["caja_tipo_id"],
        dim_interior_ancho = row["caja_interior_ancho"],
        dim_interior_largo = row["caja_interior_largo"],
        dim_interior_alto = row["caja_interior_alto"],
        grosor_mm = row["caja_grosor_mm"],
        volumen_buenos_aires = row['volumen_tipo_planta_buenos_aires'],
        volumen_curitiba = row['volumen_tipo_planta_curitiba'],
        volumen_santiago = row['volumen_tipo_planta_santiago'],
        volumen_monterrey = row['volumen_tipo_planta_monterrey'],
        volumen_bakersfield = row['volumen_tipo_planta_bakersfield'],
        costo_unitario = row['costo_unitario_base']
    )
    for _, row in caja_compras_merge.iterrows()
]

cajas[:5]

[<Caja 016d196c89dcfcb306b853a776a879b9 | Int: 248.0x383.0x224.0mm | Ext: 257.6x392.6x233.6mm | Grosor: 4.8mm>,
 <Caja 01a2a319402ed2155292c04d8748e16f | Int: 282.0x380.0x185.0mm | Ext: 291.4x389.4x194.4mm | Grosor: 4.7mm>,
 <Caja 026560e43f3fc6afe0ce89d7ddf61626 | Int: 290.0x390.0x184.0mm | Ext: 298.2x398.2x192.2mm | Grosor: 4.1mm>,
 <Caja 02d7c6680102bd11e067c00c9b71ab9c | Int: 248.0x383.0x268.0mm | Ext: 257.6x392.6x277.6mm | Grosor: 4.8mm>,
 <Caja 0378f85c226113f4ac40fd360217bb8a | Int: 289.0x390.0x224.0mm | Ext: 297.2x398.2x232.2mm | Grosor: 4.1mm>]

In [345]:
operaciones_planta_aux = operaciones_planta.drop('codigo_producto', axis=1) 
prod_op_merge = pd.concat([catalogo_productos, operaciones_planta_aux], axis=1)

productos = [
    Producto(
        codigo_producto = row['codigo_producto'],
        cantidad_paquetes = row['cantidad_paquetes'],
        peso_paquete = row['peso_neto_paquete'],
        volumen_buenos_aires = row['volumen_producto_planta_buenos_aires'],
        volumen_curitiba = row['volumen_producto_planta_curitiba'],
        volumen_santiago = row['volumen_producto_planta_santiago'],
        volumen_monterrey = row['volumen_producto_planta_monterrey'],
        volumen_bakersfield = row['volumen_producto_planta_bakersfield'],
        dim_producto_ancho = row['dim_producto_ancho'], 
        dim_producto_largo = row['dim_producto_largo'],
        dim_producto_alto = row['dim_producto_alto']
    )
    for _, row in prod_op_merge.iterrows()
]

productos[:5]

[<Producto BR0001 | Dim Prod: 285.0x386.0x303.0mm | Volumen Total: 1546613>,
 <Producto BR0002 | Dim Prod: 290.0x390.0x260.0mm | Volumen Total: 139211>,
 <Producto BR0003 | Dim Prod: 287.0x388.0x164.0mm | Volumen Total: 172506>,
 <Producto BR0004 | Dim Prod: 290.0x390.0x224.0mm | Volumen Total: 271715>,
 <Producto BR0005 | Dim Prod: 285.0x386.0x253.0mm | Volumen Total: 7586>]

#### **Factibilidad 1: Tipos de cajas asignables por dimensión**

In [346]:
cajas_asignables_dimension = {}

for producto in productos:
    cajas_asignables = []
    for caja in cajas:
        if caja.es_asignable_por_dimension(producto):
            cajas_asignables.append(caja)
    
    cajas_asignables_dimension[producto] = cajas_asignables

Veamos en detalle cuántos tipos de cajas son asignables por cada producto.

In [347]:
df_dimension = pd.DataFrame([
    {
        'codigo_producto': producto.codigo_producto,
        'cantidad_cajas_asignables': len(cajas_asign),
    }
    for producto, cajas_asign in cajas_asignables_dimension.items()
])

df_dimension = df_dimension.sort_values('cantidad_cajas_asignables', ascending=False)
df_dimension

,codigo_producto,cantidad_cajas_asignables
326,BR0317,24
325,BR0316,24
104,BR0103,22
29,BR0030,21
291,BR0284,21
...,...,...
123,BR0122,1
115,BR0114,1
57,BR0058,1
358,BR0345,1


#### **Factibilidad 2: Volumen alcanzable**

In [348]:
cajas_asignables_volumen = {}

for producto in productos:
    cajas_asignables = []
    for caja in cajas_asignables_dimension[producto]:
        if caja.volumen_total() >= producto.volumen_total():
            cajas_asignables.append(caja)
    
    cajas_asignables_volumen[producto] = cajas_asignables

In [349]:
df_volumen = pd.DataFrame([
    {
        'codigo_producto': producto.codigo_producto,
        'cantidad_cajas_asignables': len(cajas_asign),
    }
    for producto, cajas_asign in cajas_asignables_volumen.items()
])

df_volumen = df_volumen.sort_values('cantidad_cajas_asignables', ascending=False)
df_volumen

,codigo_producto,cantidad_cajas_asignables
29,BR0030,18
328,BR0319,18
347,BR0336,17
247,BR0241,17
35,BR0036,17
...,...,...
44,BR0045,1
393,BR0380,1
57,BR0058,1
6,BR0007,1


Para aquellos productos que tienen únicamente un tipo de caja asignable, les asignamos esa caja para analizar si la disminución del volumen de tipos de cajas afecta en la cantidad de cajas asignables para los demás productos.

In [350]:
productos_a_eliminar = []

for producto, cajas in cajas_asignables_volumen.items():
    if len(cajas) == 1:
        caja = cajas[0]
        caja.asignar_producto(producto)
        productos_a_eliminar.append(producto)

for producto in productos_a_eliminar:
    cajas_asignables_volumen.pop(producto)

In [351]:
for producto, cajas in cajas_asignables_volumen.items():
    cajas_asignables = []
    for caja in cajas:
        if caja.volumen_total() >= producto.volumen_total():
            cajas_asignables.append(caja)
    cajas_asignables_volumen[producto] = cajas_asignables

df_volumen = pd.DataFrame([
    {
        'codigo_producto': producto.codigo_producto,
        'cantidad_cajas_asignables': len(cajas_asign),
    }
    for producto, cajas_asign in cajas_asignables_volumen.items()
])

df_volumen = df_volumen.sort_values('cantidad_cajas_asignables', ascending=False)
df_volumen

,codigo_producto,cantidad_cajas_asignables
302,BR0319,18
27,BR0030,18
33,BR0036,17
319,BR0336,17
300,BR0317,16
...,...,...
43,BR0047,1
103,BR0108,1
285,BR0303,1
240,BR0256,1


Efectivamente, la utilización de dichos tipos de cajas hicieron que demás productos se viesen obligados a reducir su cantidad de cajas asignables.

Observemos que incluso algunos productos empezaron a poder utilizar también únicamente un tipo de caja. Iteremos nuevamente lo hecho.

In [352]:
productos_a_eliminar = []

for producto, cajas in cajas_asignables_volumen.items():
    if len(cajas) == 1:
        caja = cajas[0]
        caja.asignar_producto(producto)
        productos_a_eliminar.append(producto)

for producto in productos_a_eliminar:
    cajas_asignables_volumen.pop(producto)

In [353]:
for producto, cajas in cajas_asignables_volumen.items():
    cajas_asignables = []
    for caja in cajas:
        if caja.volumen_total() >= producto.volumen_total():
            cajas_asignables.append(caja)
    cajas_asignables_volumen[producto] = cajas_asignables

df_volumen = pd.DataFrame([
    {
        'codigo_producto': producto.codigo_producto,
        'cantidad_cajas_asignables': len(cajas_asign),
    }
    for producto, cajas_asign in cajas_asignables_volumen.items()
])

df_volumen = df_volumen.sort_values('cantidad_cajas_asignables', ascending=False)
df_volumen

,codigo_producto,cantidad_cajas_asignables
293,BR0319,18
27,BR0030,18
310,BR0336,17
33,BR0036,17
262,BR0286,16
...,...,...
326,BR0352,2
346,BR0372,2
347,BR0373,2
2,BR0004,2


In [354]:
cajas_totales = set()

for producto, cajas in cajas_asignables_volumen.items():
    for c in cajas:
        cajas_totales.add(c)

print(len(cajas_totales))

187
